# Week 6 · 문화콘텐츠 감정분석 Ⅱ
## TF-IDF와 로지스틱 회귀로 평가 절차 배우기

핵심 질문: **감정분석 모델의 성능은 어떻게 평가하고 개선할 수 있는가?**

이 자료의 120개 문장과 라벨은 **합성 교육용 예제**입니다. 실제 관객이나 플랫폼의 반응을 대표하지 않습니다.
라벨은 `0=부정`, `1=긍정`입니다. 중립을 학습한 모델이 아닙니다.

준비: README의 OS별 설치를 끝낸 뒤, VS Code 커널을 `week06/.venv`로 선택하세요.
아래 셀을 위에서 아래로 실행합니다. 기본 실행은 테스트 세트를 평가하지 않습니다.


## 1. 준비와 경로 확인
저장소 최상위 또는 week06 폴더에서 열어도 같은 CSV를 찾습니다.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

LAB = next((p for p in [Path.cwd() / "week06", Path.cwd()]
            if (p / "train_sentiment.py").exists()), None)
if LAB is None:
    raise FileNotFoundError("CNT3054 또는 week06 폴더에서 노트북을 열어주세요.")
sys.path.insert(0, str(LAB))
from train_sentiment import (
    load_data, split_data, make_model, compare_models,
    prediction_table, feature_weights, score_predictions, run_experiment,
)
print("실습 폴더:", LAB)
print("Python:", sys.version.split()[0])

## 2. 라벨은 무엇을 측정하는가?
`text`는 콘텐츠에 대한 글쓴이의 평가입니다. 작품 속 인물의 감정과 다릅니다.
서로 닮은 긍정/부정 예제쌍은 같은 `group_id`에 묶였습니다.

질문: “슬픈 결말인데도 작품 전체는 정말 훌륭했다”는 왜 1일까요?


In [ ]:
data, data_info = load_data(LAB / "data" / "synthetic_reviews.csv")
display(data[["review_id", "group_id", "domain", "text", "label"]].head(8))
display(pd.crosstab(data["domain"], data["label"]))
print(data_info)

## 3. 먼저 분리하기
학습 60%: 어휘·IDF·계수를 학습합니다. 검증 20%: 후보 비교와 개선에 씁니다. 테스트 20%: 마지막 확인에 남겨 둡니다.
같은 그룹은 절대 둘로 나누지 않으며 라벨 비율을 보존하려고 시도합니다.
도우미 함수는 StratifiedGroupKFold의 5분할에서 테스트를, 남은 4분할에서 검증을 하나씩 정합니다.
실제 그룹 크기가 불균형하면 비율은 정확히 60/20/20이 아닐 수 있습니다.

**테스트 텍스트를 읽거나 테스트로 후보를 고르지 않습니다.** 아래에서는 개수만 점검합니다.


In [ ]:
splits = split_data(data)
train = splits["train"]
validation = splits["validation"]
test = splits["test"]
summary = pd.DataFrame({name: frame["label"].value_counts().sort_index()
                        for name, frame in splits.items()})
display(summary)
assert not set(train["group_id"]) & set(validation["group_id"])
assert not set(train["group_id"]) & set(test["group_id"])
assert not set(validation["group_id"]) & set(test["group_id"])
print({name: len(frame) for name, frame in splits.items()})

## 4. 핵심 모델: TF-IDF → 로지스틱 회귀
Pipeline은 두 단계를 묶습니다. **fit은 학습 세트에서만** 호출합니다.
TF-IDF도 전체 자료에 먼저 fit하면 평가 자료의 어휘/빈도 정보가 들어가므로 누출입니다.
아래 정규식은 한 글자 `안`, `못`을 남깁니다. 형태소 분석이나 문맥 이해를 하는 것은 아닙니다.


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

unigram = Pipeline([
    ("tfidf", TfidfVectorizer(token_pattern=r"(?u)\b\w+\b", ngram_range=(1, 1))),
    ("classifier", LogisticRegression(C=1.0, max_iter=1000, random_state=42)),
])
unigram.fit(train["text"], train["label"])
validation_prediction = unigram.predict(validation["text"])
display(pd.Series(score_predictions(validation["label"], validation_prediction)))

## 5. 문장이 숫자가 된 모습
0은 해당 어휘가 그 문장에 없음을 뜻합니다. 값은 감정 점수나 긍정 확률이 아닙니다.
실제 행렬은 대부분 0인 희소 행렬입니다. 작은 일부만 표로 봅니다.


In [ ]:
vectorizer = unigram.named_steps["tfidf"]
X_train = vectorizer.transform(train["text"])
print("행(학습 문장), 열(어휘):", X_train.shape)
features = vectorizer.get_feature_names_out()
column_ids = list(range(min(10, len(features))))
display(pd.DataFrame(X_train[:5, column_ids].toarray(), columns=features[column_ids]).round(3))
print("토큰 예:", vectorizer.build_analyzer()("전혀 안 좋다"))

## 6. 기준선과 후보 비교
기준선은 학습 자료의 다수 라벨만 예측합니다. 여기서는 두 라벨 수가 같아 0을 선택합니다.
두 후보의 차이는 unigram `(1,1)`과 unigram+bigram `(1,2)`입니다.
**검증 macro-F1**로 로지스틱 회귀 후보를 선택하고, 동점이면 단순한 unigram을 택합니다.
기준선보다 나은지 반드시 함께 확인하세요. Bigram이 무조건 좋아진다고 가정하지 않습니다.


In [ ]:
models, baseline, scores, selected_name = compare_models(train, validation)
display(scores.round(4))
model = models[selected_name]
print("선택:", selected_name)
print("학습/검증을 합쳐 재학습하지 않고 이 모델을 유지합니다.")

## 7. 혼동행렬을 지표와 연결하기
순서 `[0,1]`, 행=실제, 열=예측입니다.
긍정 기준으로 왼쪽 위 TN, 오른쪽 위 FP, 왼쪽 아래 FN, 오른쪽 아래 TP입니다.
정밀도와 재현율은 분모가 다릅니다. Macro-F1은 부정 F1과 긍정 F1의 평균입니다.


In [ ]:
from sklearn.metrics import confusion_matrix

validation_result = prediction_table(model, validation)
cm = confusion_matrix(validation_result["label"], validation_result["prediction"], labels=[0, 1])
display(pd.DataFrame(cm, index=["실제 부정(0)", "실제 긍정(1)"],
                     columns=["예측 부정(0)", "예측 긍정(1)"]))
tn, fp, fn, tp = cm.ravel()
print("TN, FP, FN, TP =", tn, fp, fn, tp)
print("긍정 정밀도 =", tp / (tp + fp) if tp + fp else 0)
print("긍정 재현율 =", tp / (tp + fn) if tp + fn else 0)

## 8. 점수보다 먼저 오분류 원문 읽기
`p_positive`는 모델의 클래스 확률 출력이며 감정 강도나 정답을 보증하는 값이 아닙니다.
`known_features=0`이면 학습 어휘와 일치하는 항목이 없어 절편만으로 예측한 것입니다. 중립 판정이 아닙니다.

질문: 활용형/띄어쓰기, 학습 어휘 부족, 부정 표현, 반어, 대상 전환 중 무엇을 의심할 수 있나요?
확인하지 않은 원인을 단정하지 말고 **가설**로 기록하세요.


In [ ]:
errors = validation_result.loc[~validation_result["correct"]]
display(errors[["review_id", "text", "label", "prediction", "p_positive", "error_type", "known_features"]])

## 9. 모델은 어떤 표현을 이용했는가?
`classes_ == [0,1]`인 이 모델에서 계수가 양수이면 다른 특징이 고정될 때 긍정 log-odds 방향으로 작용합니다.
음수는 부정 방향입니다. 어휘의 보편적 감성사전 점수나 인과효과는 아닙니다.
“이야기가” 같은 보통 단어가 한쪽 방향으로 나와도 이 작은 학습 자료의 우연한 연관일 수 있습니다.


In [ ]:
weights = feature_weights(model)
display(pd.concat([weights.head(8), weights.tail(8)]).reset_index(drop=True))
print("classes_:", model.named_steps["classifier"].classes_)

## 10. 검증 결과 저장
스크립트와 같은 설정으로 다시 실행하여 결과를 새 폴더에 저장합니다.
같은 seed의 재실행은 재현성 확인이며 새로운 독립 실험이 아닙니다.


In [ ]:
output, saved_scores, metadata = run_experiment(LAB / "data" / "synthetic_reviews.csv", plots=True)
print("저장:", output)
print("테스트 평가 여부:", metadata["test_evaluated"])
assert metadata["test_evaluated"] is False
assert not list(output.glob("test_*"))

## 11. 개선 가설을 먼저 기록하세요

아래 질문에 답한 뒤에만 최종 테스트 단계로 갑니다.

1. 이 자료에서 “정답”은 무엇이며, 실제 관객의 생각과 어떤 차이가 있나요?
2. 두 후보의 검증 결과는 달랐나요? 왜 복잡한 후보가 꼭 좋아지는 것은 아닐까요?
3. 오분류 두 문장을 골라 가능한 원인과 확인 방법을 적으세요.
4. 같은 작품/작성자가 양쪽에 들어가는 실제 데이터라면 분리를 어떻게 바꾸겠습니까?
5. 긍정 Precision과 Recall 중 어느 쪽 오류가 더 중요한 연구 질문을 하나씩 적으세요.

기본 수업에서는 위 결과로 평가 절차를 설명하면 충분합니다. 매개변수를 바꾸는 확장 실험은 **검증 세트에서만** 비교하세요.
테스트를 본 뒤 바꾸려면 새 독립 평가 자료를 준비해야 합니다.


## 12. 마지막 테스트: 기본값은 False
모델·전처리·분리·선택 기준을 확정하고 교사와 확인한 뒤 `RUN_FINAL_TEST=True`로 바꾸세요.
테스트에서는 선택된 모델과 기준선을 한 번 확인합니다. 다른 후보를 테스트 점수로 다시 고르지 않습니다.
이 셀을 여러 번 실행할 수 있다고 해서 독립 평가가 여러 번 생기는 것은 아닙니다.


In [ ]:
RUN_FINAL_TEST = False
if RUN_FINAL_TEST:
    final_output, _, final_metadata = run_experiment(
        LAB / "data" / "synthetic_reviews.csv", evaluate_test=True, plots=True,
    )
    display(pd.DataFrame(final_metadata["test_scores"]).round(4))
    print("혼동행렬 [0,1]:", final_metadata["test_confusion_matrix_labels_0_1"])
    print("최종 결과 폴더:", final_output)
else:
    print("테스트는 아직 평가하지 않았습니다. 모델 선택을 먼저 마치세요.")

## 13. 결과 보고 문장 완성하기

“합성 교육용 리뷰 120개를 관련 예제쌍 기준으로 학습 72 / 검증 24 / 테스트 24개로 분리했다.
학습 세트에서만 TF-IDF와 로지스틱 회귀를 학습하고 검증 macro-F1로 후보를 선택했다.
선택된 모델은 ___이며 최종 평가를 실시했다면 Accuracy ___, 긍정 Precision ___, Recall ___, F1 ___였다.
오류 사례 ___로 보아 ___를 추가 점검해야 한다. 이 값은 실제 관객이나 다른 플랫폼의 성능 추정이 아니다.”

다음 주의 임베딩·BERT·LLM은 별도 주제입니다. 더 큰 모델을 쓰더라도 정답 라벨, 독립 평가, 오류 해석은 필요합니다.

### 공식 참고
- [TF-IDF](https://scikit-learn.org/1.8/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html)
- [LogisticRegression](https://scikit-learn.org/1.8/modules/generated/sklearn.linear_model.LogisticRegression.html)
- [누출 방지](https://scikit-learn.org/1.8/common_pitfalls.html)
- [그룹 분리](https://scikit-learn.org/1.8/modules/generated/sklearn.model_selection.StratifiedGroupKFold.html)
